Converted from feature_analysis_0407.ipynb

Notebook flow: imports, constants, dataset setup, dataloader, training utilities, training, evaluation, inference.



# Feature Analysis — `0407-onset-apex-behavior-cnn-transformer`

Notebook ini melakukan **analisa mendalam** terhadap fitur yang masuk ke proses training, validation, dan testing.
Tidak hanya visualisasi — kita akan membedah nilai vektor aktual per channel, distribusi statistik, dan memeriksa apakah normalisasi berjalan benar.

---
## Peta 47 Channel

| Range | Group | Normalisasi |
|-------|-------|-------------|
| Ch 0–19 | Flow velocity / optical flow | Z-Score |
| Ch 20–24 | Magnitude / spatial | Min-Max |
| Ch 25–34 | ROI-based flow features | Z-Score |
| Ch 35–46 | Behavioral spatial features | Min-Max |

Setiap section di bawah memeriksa satu aspek: **raw values**, **post-normalization**, **distribusi per split**, dan **deteksi anomali**.



## 0. Setup & Imports



In [ ]:
import sys
sys.path.append("..")

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from itertools import combinations
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedGroupKFold

from src.dataset.modules.window_selector import WindowSelector, ApexWindowDetector
from src.dataset.modules.behavioral_features import BehavioralFeatures
from src.dataset.modules.flow_roi_dataset import FlowROIDataset
from src.dataset.modules.compose import Compose
from src.dataset.modules.temporal_transforms import PadAndMask
from src.dataset.modules.channel_zscore import ChannelZScore
from src.dataset.modules.augment_flow import AugmentFlow

# ── Config (sama dengan notebook training) ─────────────────────────────────
MAX_SEQ_LEN   = 512
BATCH_SIZE    = 8
SEED          = 42
PHASES        = ["onset", "apex"]
TARGET_NAMES  = ["Anxiety Rendah", "Anxiety Tinggi"]
ANNOTATION_PATH = Path("/home/inadio/datasets/anxiety_raw/annotations-v5-clips.csv")
CHECKPOINT_DIR  = Path("./checkpoints_0407-onset-apex-behavior-cnn-transformer")

ZSCORE_CHANNELS = list(range(0, 20)) + list(range(25, 35))
MINMAX_CHANNELS = list(range(20, 25)) + list(range(35, 47))
N_CHANNELS = 47

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"Device: {DEVICE}")
print(f"Z-Score channels ({len(ZSCORE_CHANNELS)}): {ZSCORE_CHANNELS}")
print(f"Min-Max channels ({len(MINMAX_CHANNELS)}): {MINMAX_CHANNELS}")



## 1. Bangun Dataset & DataLoader

Kita replikasi persis split yang ada di notebook training.



In [ ]:
def collate_fn(batch):
    xs = torch.stack([item.x for item in batch])
    ys = torch.stack([item.y for item in batch])
    if batch[0].mask is not None:
        masks = torch.stack([item.mask for item in batch])
    else:
        masks = torch.zeros(len(batch), xs.shape[-1], dtype=torch.bool)
    return xs, ys, masks


def custom_sgkf_train_val_split(metadata_df, target_val_ratio=0.30, n_splits=10, random_state=42):
    work_df = metadata_df.reset_index(drop=True).copy()
    n_samples = len(work_df)
    y = work_df["label"].to_numpy()
    groups = work_df["subject_id"].to_numpy()
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_val_indices = [val_idx for _, val_idx in sgkf.split(np.zeros(n_samples), y, groups=groups)]
    n_val_folds = max(1, min(n_splits - 1, int(round(target_val_ratio * n_splits))))
    label_order = sorted(pd.unique(work_df["label"]))
    full_dist = work_df["label"].value_counts(normalize=True).reindex(label_order, fill_value=0.0).to_numpy()
    all_idx = np.arange(n_samples)
    best = None
    for fold_combo in combinations(range(n_splits), n_val_folds):
        val_idx = np.unique(np.concatenate([fold_val_indices[i] for i in fold_combo]))
        if val_idx.size == 0 or val_idx.size >= n_samples: continue
        train_mask = np.ones(n_samples, dtype=bool)
        train_mask[val_idx] = False
        train_idx = all_idx[train_mask]
        if train_idx.size == 0: continue
        val_df_c = work_df.iloc[val_idx]
        train_df_c = work_df.iloc[train_idx]
        val_dist = val_df_c["label"].value_counts(normalize=True).reindex(label_order, fill_value=0.0).to_numpy()
        score = abs(val_idx.size/n_samples - target_val_ratio) + 0.5*float(np.abs(val_dist - full_dist).sum())
        if best is None or score < best["score"]:
            best = {"score": score, "train_df": train_df_c.copy(), "val_df": val_df_c.copy()}
    return best["train_df"], best["val_df"]


# ── Load annotations ───────────────────────────────────────────────────────
norm_path = Path("scripts/annotations_normalized.csv")
df = pd.read_csv(norm_path) if norm_path.exists() else pd.read_csv(ANNOTATION_PATH)
if "npy_path" not in df.columns and "cache_path" in df.columns:
    df["npy_path"] = df["cache_path"]
if "is_valid" in df.columns:
    df = df[df["is_valid"] == True].copy()

# ── Transforms ────────────────────────────────────────────────────────────
eval_transform = Compose([
    WindowSelector(phase_includes=PHASES),
    BehavioralFeatures(),
    PadAndMask(max_len=MAX_SEQ_LEN),
    ChannelZScore(),
])

# RAW transform — tanpa ChannelZScore, untuk melihat nilai mentah
raw_transform = Compose([
    WindowSelector(phase_includes=PHASES),
    BehavioralFeatures(),
    PadAndMask(max_len=MAX_SEQ_LEN),
])

detector = ApexWindowDetector(percentile=95, prominence=0.5, max_window=MAX_SEQ_LEN)

# ── Split ─────────────────────────────────────────────────────────────────
train_df, val_df = custom_sgkf_train_val_split(df, target_val_ratio=0.30, n_splits=10)
print(f"Train: {len(train_df)} samples | Val: {len(val_df)} samples")

# ── Prototype dataset ──────────────────────────────────────────────────────
eval_proto = FlowROIDataset(metadata_df=df, detector=detector, phase_mode="onset_to_apex", transform=eval_transform)
raw_proto  = FlowROIDataset(metadata_df=df, detector=detector, phase_mode="onset_to_apex", transform=raw_transform)

def make_ds(sub_df, proto):
    kw = {a: getattr(proto, a) for a in ["detector","phase_mode","transform","roi_order","cache_dir","force_rebuild"] if hasattr(proto, a)}
    return proto.__class__(metadata_df=sub_df, **kw)

pin = torch.cuda.is_available()
train_ds_eval = make_ds(train_df, eval_proto)
train_ds_raw  = make_ds(train_df, raw_proto)
val_ds_eval   = make_ds(val_df,   eval_proto)
val_ds_raw    = make_ds(val_df,   raw_proto)

train_loader_eval = DataLoader(train_ds_eval, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)
train_loader_raw  = DataLoader(train_ds_raw,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)
val_loader_eval   = DataLoader(val_ds_eval,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)
val_loader_raw    = DataLoader(val_ds_raw,    batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)

print("DataLoaders siap.")



## 2. Helper: Kumpulkan Seluruh Vektor Fitur

Fungsi ini mengumpulkan semua nilai tensor dari satu loader ke array numpy bentuk `(N_samples, N_channels, T)`.
Kita juga simpan label dan mask.



In [ ]:
def collect_tensors(loader, max_batches=None):
    """
    Kumpulkan semua batch dari loader.
    Returns:
        X      : (N, C, T) numpy float32
        labels : (N,) numpy int
        masks  : (N, T) numpy bool
    """
    Xs, Ys, Ms = [], [], []
    for i, (x, y, m) in enumerate(loader):
        Xs.append(x.numpy())
        Ys.append(y.numpy())
        Ms.append(m.numpy())
        if max_batches and i + 1 >= max_batches:
            break
    return np.concatenate(Xs), np.concatenate(Ys), np.concatenate(Ms)

print("Mengumpulkan tensor train (raw)...")
X_train_raw, y_train, mask_train = collect_tensors(train_loader_raw)
print(f"  Shape X_train_raw : {X_train_raw.shape}  — (N_samples, C, T)")

print("Mengumpulkan tensor val (raw)...")
X_val_raw, y_val, mask_val = collect_tensors(val_loader_raw)
print(f"  Shape X_val_raw   : {X_val_raw.shape}")

print("Mengumpulkan tensor train (post-ChannelZScore)...")
X_train_norm, _, _ = collect_tensors(train_loader_eval)

print("Mengumpulkan tensor val (post-ChannelZScore)...")
X_val_norm, _, _ = collect_tensors(val_loader_eval)

print("\nSelesai mengumpulkan semua tensor.")



## 3. Statistik Per Channel — Nilai Aktual

Per channel kita hitung: mean, std, min, max, median, p5, p95.
Ini adalah nilai **vektor aktual** yang masuk ke model — bukan hanya visualisasi.



In [ ]:
def channel_stats(X, name=""):
    """
    X: (N, C, T)
    Returns DataFrame dengan statistik per channel.
    Hanya nilai non-padded (mask aware) — kita flatten seluruh (N, T) per channel.
    """
    N, C, T = X.shape
    rows = []
    X_flat = X.transpose(0, 2, 1).reshape(-1, C)  # (N*T, C)
    for c in range(C):
        vals = X_flat[:, c]
        group = "z-score" if c in ZSCORE_CHANNELS else "min-max"
        rows.append({
            "channel": c,
            "group": group,
            "mean": float(np.mean(vals)),
            "std":  float(np.std(vals)),
            "min":  float(np.min(vals)),
            "p5":   float(np.percentile(vals, 5)),
            "median": float(np.median(vals)),
            "p95":  float(np.percentile(vals, 95)),
            "max":  float(np.max(vals)),
            "n_zero_pct": float(np.mean(vals == 0) * 100),
            "n_nan": int(np.sum(np.isnan(vals))),
            "n_inf": int(np.sum(np.isinf(vals))),
        })
    df_stats = pd.DataFrame(rows)
    df_stats["split"] = name
    return df_stats

stats_train_raw  = channel_stats(X_train_raw,  name="train_raw")
stats_val_raw    = channel_stats(X_val_raw,    name="val_raw")
stats_train_norm = channel_stats(X_train_norm, name="train_norm")
stats_val_norm   = channel_stats(X_val_norm,   name="val_norm")

# Tampilkan ringkasan
print("=" * 80)
print("STATISTIK PER CHANNEL — TRAIN RAW (sebelum normalisasi)")
print("=" * 80)
pd.set_option("display.float_format", "{:.6f}".format)
display(stats_train_raw.drop(columns=["split"]).to_string(index=False))

print("\n" + "=" * 80)
print("STATISTIK PER CHANNEL — TRAIN POST-ChannelZScore")
print("=" * 80)
display(stats_train_norm.drop(columns=["split"]).to_string(index=False))



## 4. Nilai Vektor Sample-by-Sample (Batch Pertama)

Kita periksa **nilai aktual tensor** dari 3 sample pertama — sebelum dan sesudah normalisasi, per channel.



In [ ]:
def print_sample_vector(X, label, y, sample_idx=0, t_range=(0, 10), channels=None):
    """
    Cetak nilai aktual tensor sample ke-`sample_idx` untuk channel tertentu
    pada time step t_range.
    X: (N, C, T)
    """
    if channels is None:
        channels = list(range(X.shape[1]))
    t0, t1 = t_range
    x = X[sample_idx]  # (C, T)
    class_name = TARGET_NAMES[int(y[sample_idx])]
    print(f"\n{'─'*70}")
    print(f"Sample idx={sample_idx} | Label: {class_name} | Shape: {x.shape}")
    print(f"Time steps ditampilkan: t={t0} sampai t={t1-1}")
    print(f"{'─'*70}")
    print(f"{'Ch':>4} {'Group':>8}  {'t'+str(t0):>10}", end="")
    for t in range(t0+1, t1):
        print(f"  {'t'+str(t):>10}", end="")
    print()
    print("-" * (4 + 10 + (t1-t0)*12))
    for c in channels:
        grp = "z-scr" if c in ZSCORE_CHANNELS else "mm"
        vals = x[c, t0:t1]
        print(f"  {c:>2} {grp:>6}  ", end="")
        for v in vals:
            print(f"  {v:>10.5f}", end="")
        print()

# ── Tampilkan 3 sample pertama, channel 0-9 dan 20-24 ─────────────────────
channels_to_show = list(range(0, 10)) + list(range(20, 25))

print("=" * 70)
print("NILAI VEKTOR AKTUAL — RAW (sebelum normalisasi)")
print("=" * 70)
for i in range(min(3, len(X_train_raw))):
    print_sample_vector(X_train_raw, "raw", y_train, sample_idx=i,
                        t_range=(0, 10), channels=channels_to_show)

print("\n" + "=" * 70)
print("NILAI VEKTOR AKTUAL — POST-ChannelZScore")
print("=" * 70)
for i in range(min(3, len(X_train_norm))):
    print_sample_vector(X_train_norm, "norm", y_train, sample_idx=i,
                        t_range=(0, 10), channels=channels_to_show)



## 5. Fit & Inspeksi `hybrid_stats`

`hybrid_stats` adalah statistik yang dihitung dari train set dan digunakan untuk normalisasi saat inference.
Kita tampilkan nilai aktual mu, std, xmin, xmax per channel.



In [ ]:
def fit_hybrid_stats(loader):
    sum_c = torch.zeros(N_CHANNELS, dtype=torch.float64)
    sq_c  = torch.zeros(N_CHANNELS, dtype=torch.float64)
    min_c = torch.full((N_CHANNELS,), float("inf"), dtype=torch.float64)
    max_c = torch.full((N_CHANNELS,), float("-inf"), dtype=torch.float64)
    n_obs = 0
    for batch_x, _, _ in loader:
        x = batch_x.detach().cpu().to(torch.float64)
        flat = x.permute(0, 2, 1).reshape(-1, x.shape[1])
        if flat.numel() == 0: continue
        sum_c += flat.sum(dim=0)
        sq_c  += (flat * flat).sum(dim=0)
        min_c = torch.minimum(min_c, flat.min(dim=0).values)
        max_c = torch.maximum(max_c, flat.max(dim=0).values)
        n_obs += flat.shape[0]
    mu  = sum_c / n_obs
    var = torch.clamp(sq_c / n_obs - mu * mu, min=1e-12)
    std = torch.sqrt(var)
    return {"mu": mu.float(), "std": std.float(), "xmin": min_c.float(), "xmax": max_c.float()}

# Fit dari train loader eval (setelah ChannelZScore, seperti di training asli)
# Note: di training asli hybrid stats dihitung dari train_loader_clean (eval transform)
hybrid_stats = fit_hybrid_stats(train_loader_raw)  # dari raw agar lebih informatif

# Susun ke DataFrame
hs_df = pd.DataFrame({
    "channel":  list(range(N_CHANNELS)),
    "group":    ["z-score" if c in ZSCORE_CHANNELS else "min-max" for c in range(N_CHANNELS)],
    "mu":       hybrid_stats["mu"].numpy(),
    "std":      hybrid_stats["std"].numpy(),
    "xmin":     hybrid_stats["xmin"].numpy(),
    "xmax":     hybrid_stats["xmax"].numpy(),
    "range":    (hybrid_stats["xmax"] - hybrid_stats["xmin"]).numpy(),
})

print("=" * 80)
print("HYBRID STATS — Nilai Aktual Per Channel (dihitung dari train set raw)")
print("=" * 80)
pd.set_option("display.float_format", "{:.8f}".format)
display(hs_df)

# Cek channel dengan range mendekati nol (potensi masalah normalisasi)
low_range = hs_df[hs_df["range"] < 1e-4]
if len(low_range) > 0:
    print(f"\n⚠️  Channel dengan range < 1e-4 (risiko division by zero di min-max):")
    display(low_range)
else:
    print("\n✓ Semua channel memiliki range > 1e-4 — aman untuk min-max normalisasi.")



## 6. Distribusi Nilai Per Channel — Visualisasi Heatmap

Kita bandingkan distribusi mean per channel antara train dan val set.



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle("Distribusi Per Channel — Nilai Aktual (Raw vs Post-ChannelZScore)", fontsize=14, fontweight="bold")

def plot_channel_stats(ax, stats_df, title, metric="mean", color="steelblue"):
    channels = stats_df["channel"].values
    vals = stats_df[metric].values
    colors = ["#2196F3" if c in ZSCORE_CHANNELS else "#FF5722" for c in channels]
    bars = ax.bar(channels, vals, color=colors, alpha=0.8, width=0.8)
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Channel Index")
    ax.set_ylabel(metric)
    ax.set_xlim(-1, N_CHANNELS)
    # Tambahkan garis pembatas antar grup
    for boundary in [20, 25, 35]:
        ax.axvline(boundary - 0.5, color="black", linewidth=1, linestyle=":")
    from matplotlib.patches import Patch
    legend_elems = [
        Patch(facecolor="#2196F3", label="Z-Score group"),
        Patch(facecolor="#FF5722", label="Min-Max group")
    ]
    ax.legend(handles=legend_elems, loc="upper right", fontsize=8)
    ax.tick_params(axis="x", labelsize=7)

plot_channel_stats(axes[0, 0], stats_train_raw,  "Train RAW — Mean per Channel",  metric="mean")
plot_channel_stats(axes[0, 1], stats_train_raw,  "Train RAW — Std per Channel",   metric="std",  color="#FF5722")
plot_channel_stats(axes[1, 0], stats_train_norm, "Train POST-ZScore — Mean per Channel", metric="mean")
plot_channel_stats(axes[1, 1], stats_train_norm, "Train POST-ZScore — Std per Channel",  metric="std",  color="#FF5722")

plt.tight_layout()
plt.savefig("feature_channel_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: feature_channel_distribution.png")



## 7. Perbandingan Train vs Val — Distributional Shift

Kita ukur **distributional shift** antara train dan val per channel menggunakan Mean Absolute Difference.
Ini penting untuk memastikan tidak ada kebocoran data atau distribusi yang sangat berbeda.



In [ ]:
# Merge stats
shift_df = stats_train_raw[["channel", "group", "mean", "std"]].copy()
shift_df = shift_df.rename(columns={"mean": "train_mean", "std": "train_std"})
shift_df["val_mean"]  = stats_val_raw["mean"].values
shift_df["val_std"]   = stats_val_raw["std"].values
shift_df["mean_diff"] = np.abs(shift_df["train_mean"] - shift_df["val_mean"])
shift_df["std_diff"]  = np.abs(shift_df["train_std"]  - shift_df["val_std"])
shift_df["mean_shift_pct"] = shift_df["mean_diff"] / (np.abs(shift_df["train_mean"]) + 1e-8) * 100

# Tampilkan channel dengan shift paling besar
print("=" * 80)
print("DISTRIBUTIONAL SHIFT — Train vs Val per Channel (Raw)")
print("=" * 80)
pd.set_option("display.float_format", "{:.6f}".format)
display(shift_df.sort_values("mean_diff", ascending=False).head(20))

# Channel dengan shift paling mengkhawatirkan
HIGH_SHIFT_THRESHOLD = 0.1
high_shift = shift_df[shift_df["mean_diff"] > HIGH_SHIFT_THRESHOLD]
print(f"\nChannel dengan |mean_diff| > {HIGH_SHIFT_THRESHOLD}:")
if len(high_shift) > 0:
    display(high_shift)
else:
    print(f"  ✓ Tidak ada — shift terkontrol di semua channel.")

# Plot
fig, ax = plt.subplots(figsize=(16, 4))
colors = ["#2196F3" if c in ZSCORE_CHANNELS else "#FF5722" for c in shift_df["channel"]]
ax.bar(shift_df["channel"], shift_df["mean_diff"], color=colors, alpha=0.8)
ax.axhline(HIGH_SHIFT_THRESHOLD, color="red", linestyle="--", linewidth=0.8, label=f"Threshold {HIGH_SHIFT_THRESHOLD}")
for boundary in [20, 25, 35]:
    ax.axvline(boundary - 0.5, color="black", linewidth=1, linestyle=":")
ax.set_title("Distributional Shift (|mean_train - mean_val|) per Channel", fontsize=12)
ax.set_xlabel("Channel")
ax.set_ylabel("|Δ mean|")
ax.legend()
plt.tight_layout()
plt.savefig("feature_shift_train_val.png", dpi=150, bbox_inches="tight")
plt.show()



## 8. Temporal Profile — Nilai Vektor Sepanjang Waktu

Untuk beberapa channel representatif, kita plot rata-rata nilai per time step (averaged over samples),
dibedakan per kelas (Anxiety Rendah vs Tinggi).
Ini mengungkap apakah ada **temporal pattern** yang ditangkap fitur.



In [ ]:
def plot_temporal_profile(X, y, mask, channels_to_plot, title_prefix="Train", max_t=200):
    """
    Plot rata-rata nilai per time step per kelas, untuk channel tertentu.
    X: (N, C, T), y: (N,), mask: (N, T) — True = padding
    """
    n_ch = len(channels_to_plot)
    fig, axes = plt.subplots(2, (n_ch + 1) // 2, figsize=(16, 6))
    axes = axes.flatten()
    fig.suptitle(f"{title_prefix} — Temporal Profile per Channel (mean ± std per kelas)", fontsize=12)

    class_0_idx = np.where(y == 0)[0]
    class_1_idx = np.where(y == 1)[0]

    t = np.arange(max_t)

    for ax_i, c in enumerate(channels_to_plot):
        ax = axes[ax_i]
        grp = "Z-Score" if c in ZSCORE_CHANNELS else "Min-Max"

        for cls_idx, cls_name, color in [
            (class_0_idx, "Anxiety Rendah", "#2196F3"),
            (class_1_idx, "Anxiety Tinggi", "#F44336"),
        ]:
            vals = X[cls_idx, c, :max_t]  # (N_cls, max_t)
            # Masker padding: mask=True berarti padding
            mk = mask[cls_idx, :max_t].astype(bool)  # True = padding
            # Hitung mean & std hanya pada non-padded steps
            valid_counts = (~mk).sum(axis=0)  # (max_t,)
            vals_masked = np.where(mk, np.nan, vals)
            mean_t = np.nanmean(vals_masked, axis=0)
            std_t  = np.nanstd(vals_masked, axis=0)

            ax.plot(t, mean_t, color=color, linewidth=1.2, label=cls_name)
            ax.fill_between(t, mean_t - std_t, mean_t + std_t, alpha=0.15, color=color)

        ax.set_title(f"Ch {c} [{grp}]", fontsize=9)
        ax.set_xlabel("Time step", fontsize=7)
        ax.set_ylabel("Value", fontsize=7)
        ax.tick_params(labelsize=7)
        if ax_i == 0:
            ax.legend(fontsize=7)

    for ax_i in range(len(channels_to_plot), len(axes)):
        axes[ax_i].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"feature_temporal_{title_prefix.lower().replace(' ', '_')}.png",
                dpi=150, bbox_inches="tight")
    plt.show()

# Channel representatif untuk di-plot
# Z-Score group: 0, 5, 10, 15, 25, 30
# Min-Max group: 20, 22, 35, 40
channels_representative = [0, 5, 10, 15, 20, 22, 25, 30, 35, 40]

plot_temporal_profile(X_train_raw, y_train, mask_train,
                      channels_to_plot=channels_representative,
                      title_prefix="Train Raw")

plot_temporal_profile(X_train_norm, y_train, mask_train,
                      channels_to_plot=channels_representative,
                      title_prefix="Train Post-ZScore")



## 9. Analisa Padding & Sequence Length

Berapa persen time step yang sebenarnya berisi data vs padding?
Penting untuk mengetahui effective sequence length per sample.



In [ ]:
def analyze_padding(mask, split_name):
    """
    mask: (N, T) — True = padding
    """
    N, T = mask.shape
    # Effective length = jumlah non-padding per sample
    eff_len = (~mask.astype(bool)).sum(axis=1)  # (N,)

    print(f"\n{'='*60}")
    print(f"PADDING ANALYSIS — {split_name}")
    print(f"{'='*60}")
    print(f"  N samples         : {N}")
    print(f"  Max seq len (T)   : {T}")
    print(f"  Effective length  :")
    print(f"    mean  = {eff_len.mean():.1f}")
    print(f"    std   = {eff_len.std():.1f}")
    print(f"    min   = {eff_len.min()}")
    print(f"    max   = {eff_len.max()}")
    print(f"    median= {np.median(eff_len):.0f}")
    print(f"  % padding overall : {mask.mean()*100:.1f}%")
    print(f"  Samples at T_max  : {(eff_len == T).sum()} ({(eff_len==T).mean()*100:.1f}%)")

    # Histogram
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.hist(eff_len, bins=30, edgecolor="white", color="#2196F3", alpha=0.8)
    ax.axvline(eff_len.mean(), color="red", linestyle="--", linewidth=1.2, label=f"mean={eff_len.mean():.0f}")
    ax.set_title(f"{split_name} — Distribusi Effective Sequence Length")
    ax.set_xlabel("Effective Time Steps (non-padded)")
    ax.set_ylabel("Count")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"feature_seqlen_{split_name.lower()}.png", dpi=120, bbox_inches="tight")
    plt.show()
    return eff_len

eff_train = analyze_padding(mask_train, "Train")
eff_val   = analyze_padding(mask_val,   "Val")



## 10. Inspeksi Fitur Behavioral Per Kelas — Box Plot

Per channel, kita plot distribusi nilai (averaged over T) per sample, dibedakan per kelas.
Ini menunjukkan channel mana yang paling diskriminatif.



In [ ]:
def plot_per_class_boxplot(X, y, mask, split_name="Train", n_channels_show=47):
    """
    Per channel, hitung rata-rata nilai per sample (hanya non-padding),
    lalu boxplot per kelas.
    """
    N, C, T = X.shape
    # Mean per sample per channel (ignore padding)
    mk = mask.astype(bool)  # (N, T) True=padding
    # Expand mask ke shape (N, C, T)
    mk_exp = np.broadcast_to(mk[:, np.newaxis, :], (N, C, T))
    X_masked = np.where(mk_exp, np.nan, X)
    sample_means = np.nanmean(X_masked, axis=2)  # (N, C)

    # Hitung t-statistic per channel (ukuran discriminability)
    from scipy import stats as sp_stats
    discriminability = []
    for c in range(C):
        g0 = sample_means[y == 0, c]
        g1 = sample_means[y == 1, c]
        if len(g0) > 1 and len(g1) > 1:
            tstat, pval = sp_stats.ttest_ind(g0, g1, equal_var=False)
        else:
            tstat, pval = 0.0, 1.0
        discriminability.append({"channel": c, "t_stat": tstat, "p_value": pval,
                                  "abs_t": abs(tstat)})
    disc_df = pd.DataFrame(discriminability).sort_values("abs_t", ascending=False)

    print(f"\n{'='*60}")
    print(f"TOP 10 CHANNEL PALING DISKRIMINATIF — {split_name}")
    print(f"(Welch's t-test, class 0 vs 1)")
    print(f"{'='*60}")
    pd.set_option("display.float_format", "{:.5f}".format)
    display(disc_df.head(10))

    # Plot top 12 channel diskriminatif
    top_channels = disc_df["channel"].values[:12]
    fig, axes = plt.subplots(3, 4, figsize=(16, 9))
    axes = axes.flatten()
    fig.suptitle(f"{split_name} — Box Plot Per Kelas (Top-12 Most Discriminative Channels)", fontsize=12)

    for ax_i, c in enumerate(top_channels):
        ax = axes[ax_i]
        data = [
            sample_means[y == 0, c],
            sample_means[y == 1, c],
        ]
        bp = ax.boxplot(data, patch_artist=True,
                        medianprops={"color": "white", "linewidth": 2})
        bp["boxes"][0].set_facecolor("#2196F3")
        bp["boxes"][1].set_facecolor("#F44336")
        grp = "Z" if c in ZSCORE_CHANNELS else "MM"
        t_row = disc_df[disc_df["channel"] == c].iloc[0]
        ax.set_title(f"Ch {c} [{grp}] | t={t_row['t_stat']:.2f}", fontsize=8)
        ax.set_xticks([1, 2])
        ax.set_xticklabels(["Rendah", "Tinggi"], fontsize=7)
        ax.tick_params(axis="y", labelsize=7)

    for ax_i in range(len(top_channels), len(axes)):
        axes[ax_i].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"feature_boxplot_{split_name.lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()
    return disc_df

disc_train = plot_per_class_boxplot(X_train_raw, y_train, mask_train, split_name="Train")
disc_val   = plot_per_class_boxplot(X_val_raw,   y_val,   mask_val,   split_name="Val")



## 11. Inspeksi Fitur Test Set (Eksternal)

Kita muat test set dan bandingkan distribusinya terhadap train set.



In [ ]:
LABEL_MAP = {"anxiety_rendah": 0, "anxiety_tinggi": 1}

for test_name, test_path in [
    ("Test-Ext1", Path("/home/inadio/datasets/dataset_test/clips-annotations-detail.csv")),
    ("Test-Ext2", Path("/home/inadio/datasets/dataset_test_2/annotations-clips.csv")),
]:
    if not test_path.exists():
        print(f"[Skip] {test_name} — file tidak ditemukan: {test_path}")
        continue

    df_test = pd.read_csv(test_path)
    if "npy_path" not in df_test.columns and "cache_path" in df_test.columns:
        df_test["npy_path"] = df_test["cache_path"]
    if "is_valid" in df_test.columns:
        df_test = df_test[df_test["is_valid"] == True].copy()
    df_test = df_test[df_test["label"].isin(LABEL_MAP)].copy()
    print(f"\n{test_name}: {len(df_test)} samples, {df_test['subject_id'].nunique()} subjects")

    test_ds = FlowROIDataset(
        metadata_df=df_test, detector=detector,
        phase_mode="full", transform=raw_transform,
    )
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=0, collate_fn=collate_fn)

    print(f"  Mengumpulkan tensor {test_name}...")
    X_test, y_test, mask_test = collect_tensors(test_loader)
    print(f"  Shape: {X_test.shape}")

    stats_test = channel_stats(X_test, name=test_name)

    # Perbandingan mean per channel: train vs test
    fig, ax = plt.subplots(figsize=(16, 4))
    ch = np.arange(N_CHANNELS)
    ax.plot(ch, stats_train_raw["mean"].values, label="Train", color="#2196F3", linewidth=1.2)
    ax.plot(ch, stats_test["mean"].values, label=test_name, color="#F44336", linewidth=1.2, linestyle="--")
    for boundary in [20, 25, 35]:
        ax.axvline(boundary - 0.5, color="gray", linewidth=0.8, linestyle=":")
    ax.set_title(f"Mean per Channel: Train vs {test_name}", fontsize=11)
    ax.set_xlabel("Channel")
    ax.set_ylabel("Mean value")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"feature_shift_train_vs_{test_name.lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Statistik lengkap
    print(f"\nStatistik per channel — {test_name}:")
    display(stats_test)

    # Padding analysis
    analyze_padding(mask_test, test_name)



## 12. Export Ringkasan ke CSV

Export semua statistik ke file CSV agar bisa dibuka di Excel atau tool lain.



In [ ]:
# Gabungkan semua stats ke satu file
all_stats = pd.concat([
    stats_train_raw,
    stats_val_raw,
    stats_train_norm,
    stats_val_norm,
], ignore_index=True)

out_path = Path("feature_analysis_stats.csv")
all_stats.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

# Export hybrid stats
hs_path = Path("feature_hybrid_stats.csv")
hs_df.to_csv(hs_path, index=False)
print(f"Saved: {hs_path}")

# Export discriminability
disc_path = Path("feature_discriminability.csv")
disc_train.to_csv(disc_path, index=False)
print(f"Saved: {disc_path}")

print("\n=== RINGKASAN AKHIR ===")
print(f"Total channel   : {N_CHANNELS}")
print(f"Z-Score channels: {len(ZSCORE_CHANNELS)} (ch {ZSCORE_CHANNELS[:3]}...{ZSCORE_CHANNELS[-3:]})")
print(f"Min-Max channels: {len(MINMAX_CHANNELS)} (ch {MINMAX_CHANNELS[:3]}...{MINMAX_CHANNELS[-3:]})")
print(f"Train samples   : {len(X_train_raw)}")
print(f"Val samples     : {len(X_val_raw)}")
print(f"Seq length (T)  : {MAX_SEQ_LEN}")
print(f"Total params analyzed: {len(X_train_raw) * N_CHANNELS * MAX_SEQ_LEN:,} values")



In [ ]:
# === Test Inference Function ===

def run_test_inference(annotation_path, output_tag, label_map=LABEL_MAP, threshold=0.5, title=None):
    """
    Run inference on a test set and display metrics.
    This function is adapted from the training notebook 0407 for analysis purposes.
    """
    annotation_path = Path(annotation_path)
    if not annotation_path.exists():
        print(f"[Skip] Anotasi tidak ditemukan: {annotation_path}")
        return None

    df_test = pd.read_csv(annotation_path)
    if "npy_path" not in df_test.columns and "cache_path" in df_test.columns:
        df_test["npy_path"] = df_test["cache_path"]
    if "is_valid" in df_test.columns:
        df_test = df_test[df_test["is_valid"]].copy()
    df_test = df_test[df_test["label"].isin(label_map)].copy()
    print(f"[Test] {len(df_test)} clips, {df_test['subject_id'].nunique()} subjects")
    
    # Build test dataset and loader
    feature_transform = BehavioralFeatures()
    detector = ApexWindowDetector(
        percentile=DETECTOR_PERCENTILE if 'DETECTOR_PERCENTILE' in globals() else 90,
        prominence=DETECTOR_PROMINENCE if 'DETECTOR_PROMINENCE' in globals() else 0.1,
        max_window=MAX_SEQ_LEN if 'MAX_SEQ_LEN' in globals() else 512,
    )
    
    eval_transform = Compose([
        WindowSelector(phase_includes=PHASES),
        feature_transform,
        PadAndMask(max_len=MAX_SEQ_LEN if 'MAX_SEQ_LEN' in globals() else 512),
        AugmentFlow(training=False),
    ])
    
    def _collate_fn(batch):
        xs = torch.stack([item.x for item in batch])
        ys = torch.stack([item.y for item in batch])
        masks = (
            torch.stack([item.mask for item in batch])
            if batch[0].mask is not None
            else torch.zeros(len(batch), xs.shape[-1], dtype=torch.bool)
        )
        return xs, ys, masks
    
    ds = FlowROIDataset(
        metadata_df=df_test,
        detector=detector,
        phase_mode="full",
        transform=eval_transform,
    )
    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE if 'BATCH_SIZE' in globals() else 8,
        shuffle=False,
        num_workers=0,
        collate_fn=_collate_fn,
    )
    
    # Load model and checkpoint
    display_name = title or annotation_path.name
    print(f"\n{'=' * 60}")
    print(f"[Test Analysis — {display_name}]")
    print(f"{'=' * 60}")
    
    # Collect feature statistics from test set
    X_test, y_test, masks_test = collect_tensors(loader)
    print(f"\nTest set features shape: {X_test.shape}")
    print(f"  - Samples: {X_test.shape[0]}")
    print(f"  - Channels: {X_test.shape[1]}")
    print(f"  - Time steps: {X_test.shape[2]}")
    print(f"  - Class distribution: {np.bincount(y_test)}")
    
    stats_test = channel_stats(X_test, name=f"{display_name}_test")
    print(f"\nTest set channel stats:")
    print(stats_test.describe())
    
    return {"X": X_test, "y": y_test, "masks": masks_test, "stats": stats_test}


# Run inference on both test sets
LABEL_MAP = {"anxiety_rendah": 0, "anxiety_tinggi": 1}

test_results = {}
for test_name, test_path in [
    ("Test-Ext1-Mahasiswa", Path("/home/inadio/datasets/dataset_test/clips-annotations-detail.csv")),
    ("Test-Ext2-SMK", Path("/home/inadio/datasets/dataset_test_2/annotations-clips.csv")),
]:
    result = run_test_inference(test_path, output_tag=test_name, title=test_name)
    if result is not None:
        test_results[test_name] = result
        print(f"\n✓ Completed {test_name}")
    
print(f"\n{'=' * 60}")
print(f"Completed test inference on {len(test_results)} external test sets")
print(f"{'=' * 60}")